In [1]:
from go_ml.models.bert_finetune import BERTFinetune
from transformers import AutoTokenizer, AutoConfig

import torch
device = torch.device('cuda:0')
checkpoint_dir = "/home/andrew/GO_interp/checkpoints"
# with open(f"{checkpoint_dir}/esm_finetune_hparams.pkl", "rb") as f:
#     hparams = pickle.load(f)
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model = BERTFinetune.load_from_checkpoint(f"{checkpoint_dir}/esm_finetune-v1.ckpt", map_location=device)
model.eval()
print('model loaded')

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model loaded


In [ ]:
import json, pickle
from go_ml.data_utils import *
train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)
with open(f"{train_path}/cafa_dataset/prot_ids.json", "r") as f:
    prot_ids = json.load(f)
with open(f"{train_path}/cafa_dataset/rev_annot.pkl", "rb") as f:
    labels = pickle.load(f)
print((np.asarray(labels.sum(axis=1)) > 0).sum() / labels.shape[0])
prot_sequences, seq_ids = load_protein_sequences(f"{train_path}/uniprot_sprot.fasta")

# print(len(seq_ids), seq_ids[:100])
# seq_ids = [s.split("|")[1] for s in seq_ids]
assert all(s1 == s2 for s1, s2 in zip(prot_ids, seq_ids))
labeled_id = (np.asarray(labels.sum(axis=1)) > 0).flatten()

gen = np.random.default_rng(seed=42)
ind = gen.permuted(np.arange(len(prot_ids))[labeled_id])
train_len = ind.shape[0] * 4 // 5
train_ind = ind[:train_len]
val_ind = ind[train_len:]
np.sort(train_ind); np.sort(val_ind)
train_ids = [prot_ids[i] for i in train_ind]; train_sequences = [prot_sequences[i] for i in train_ind]
val_ids = [prot_ids[i] for i in val_ind]; val_sequences = [prot_sequences[i] for i in val_ind]
train_labels = labels[train_ind, :]; val_labels = labels[val_ind, :]
train_dataset = BertSeqDataset(train_ids, go_terms, train_sequences, train_labels)
val_dataset = BertSeqDataset(val_ids, go_terms, val_sequences, val_labels)
print(f"train len {len(train_dataset)} val len {len(val_dataset)}")

0.27317815417177815
train len 124524 val len 31131


In [ ]:
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
collate_seqs = get_seq_collator(tokenizer, max_length=1024, add_special_tokens=True)
val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn":collate_seqs}
train_loader = DataLoader(train_dataset, **val_dataloader_params)
val_loader = DataLoader(val_dataset, **val_dataloader_params)
y_hat_l = []
with torch.no_grad():
    for batch in tqdm(train_loader):
        # print(batch.keys())
        inputs, mask, y = batch['seq_ind'].to(device), batch['mask'].to(device), batch['labels']
        y_hat = model.forward(inputs, mask)
        y_hat_l.append(y_hat.cpu())
logits = torch.concat(y_hat_l)
print(logits.shape)


  0%|          | 0/10377 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print(logits.shape)
from sklearn.metrics import f1_score
preds = logits > 0

torch.Size([28908, 29185])


In [23]:
labels = train_labels[:pi_pred.shape[0]] # torch.Size([11472, 29185])
f1 = f1_score(labels, preds[:pi_pred.shape[0]], average='macro')

/home/andrew/anaconda3/envs/esm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [24]:
print(f1)

0.12882967371899304


In [ ]:
import onnxruntime as rt
rt.get_device()
import json
train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)
with open('/home/andrew/GO_interp/data/proteinf_go_terms.json', 'r') as f:
    go_terms_pi = json.load(f)

sess = rt.InferenceSession("/home/andrew/GO_interp/data/proteinf_model.onnx")
import pickle
AMINO_ACID_VOCABULARY = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
from collections import defaultdict
RESIDUE_TO_INT = defaultdict(lambda : 0, {aa: idx for idx, aa in enumerate(AMINO_ACID_VOCABULARY)})
import numpy as np
oh_mat = np.eye(len(RESIDUE_TO_INT), dtype=np.float32)
def get_onehot(seq_ind):
    seq_oh = oh_mat[seq_ind]
    return seq_oh
def get_seq_logit(seq):
    seq_len = len(seq)
    batch_seq = np.array([RESIDUE_TO_INT[c] for c in seq]).reshape((1, -1))
    batch_len = np.full((1,), seq_len, dtype=np.int32)
    batch_seq_oh = get_onehot(batch_seq)
    logits = sess.run(output_names=['output'], input_feed={'sequence_length': batch_len, 'sequence': batch_seq_oh})
    return logits

pi_logit_l = []
for seq in tqdm(train_sequences):
    pi_logit = get_seq_logit(seq)
    pi_logit_l.append(pi_logit)

In [12]:
pi_logits = np.concatenate([x[0] for x in pi_logit_l][:11472])
from go_ml.go_utils import godag, go2parents_isa, get_ancestors
train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)
with open('/home/andrew/GO_interp/data/proteinf_go_terms.json', 'r') as f:
    go_terms_pi = json.load(f)
    
len(go_terms), len(go_terms_pi), len(set(go_terms).intersection(go_terms_pi))

../go-basic.obo: fmt(1.2) rel(2024-04-24) 45,667 Terms


(29185, 32102, 26832)

In [14]:
pi_term_map = defaultdict(lambda:29183, {gt:i for i, gt in enumerate(go_terms_pi)})
pi_term_ind = [pi_term_map[gt] for gt in go_terms]
pi_logits_remap = pi_logits[:, pi_term_ind]
pi_pred = pi_logits_remap > 0.5

In [20]:
labels = train_labels[:pi_pred.shape[0]] # torch.Size([11472, 29185])
f1 = f1_score(labels, pi_pred, average='macro')
print(f1)

0.35539443389138475


/home/andrew/anaconda3/envs/esm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[3172,
 3041,
 3286,
 3049,
 7690,
 17786,
 3107,
 3664,
 108,
 8161,
 3252,
 1895,
 3068,
 1874,
 8200,
 3244,
 9292,
 7984,
 1902,
 31095,
 3173,
 1916,
 3050,
 3574,
 174,
 3117,
 8267,
 5684,
 8230,
 4749,
 5007,
 2633,
 7884,
 3731,
 3624,
 4585,
 7889,
 20031,
 20034,
 3212,
 3632,
 33,
 4674,
 8278,
 3177,
 4093,
 3347,
 3450,
 3980,
 5524,
 8250,
 28370,
 5707,
 62,
 3840,
 3599,
 7087,
 11375,
 1873,
 1995,
 3222,
 8193,
 17831,
 3097,
 19889,
 4667,
 7668,
 20906,
 1888,
 4725,
 3842,
 4142,
 1872,
 3218,
 3264,
 2524,
 4212,
 3669,
 3556,
 7671,
 3697,
 2396,
 15159,
 3745,
 8177,
 5706,
 1904,
 1976,
 3666,
 5094,
 5361,
 17837,
 8166,
 3536,
 17153,
 3037,
 3665,
 3168,
 14623,
 3620,
 14762,
 1903,
 5732,
 4666,
 105,
 2397,
 23467,
 25119,
 3626,
 4229,
 9761,
 3654,
 3148,
 4820,
 17109,
 3495,
 15008,
 13784,
 19707,
 5518,
 2209,
 3449,
 10099,
 16712,
 3067,
 15455,
 1906,
 16258,
 9395,
 3957,
 5381,
 5346,
 29183,
 3943,
 5009,
 3453,
 5343,
 10608,
 2720,
 501,
 9

0.8211118773465962
